# Deepfake Audio Detection — F-SAT Replication

By: Jazmin Huerta  
CS 5352 - Computer Security  
Professor Aritran Piplai

**I used claude to guide me in setting the project up, and as a help resource.**

Replication of Table 3 from:
> *"I Can Hear You: Selective Robust Training for Deepfake Audio Detection"*  
> Zhang et al., ICLR 2025

### What this notebook does
Trains **3 model variants** and evaluates them as closely to the paper as possible:
| Model | What it adds |
|---|---|
| `RawNet3` (baseline) | Plain cross-entropy, no augmentation |
| `RawNet3 + RandAug` | Random audio augmentations during training |
| `RawNet3 + RandAug + F-SAT` | + Frequency-selective adversarial training (4–8kHz) |

Evaluated on:
- **ASVspoof 2019** — in-distribution (same synthesis methods as training)
- **FakeAudio (Kaggle)** — out-of-distribution (newer synthesis methods)

---
### BEFORE you start!!!
1. `Runtime` → `Change runtime type` → **A100 GPU**
2. Run Cell 1 to mount your Google Drive
3. **Option A** First time: run Cell 2 to download datasets from Kaggle (~55 min) they will also save to Drive
4. **Returning user?** Skip Cell 2 entirely, datasets read directly from Drive
5. Run all remaining cells in order (1-12 skipping no.2 if you are a returning user)

## Cell 1 — Mount Drive & Install Packages

Mounts your Google Drive and installs required packages.

> **Drive is used for two things:**
> 1. Storing the datasets (so you never re-download them)
> 2. Saving model checkpoints, results, and plots automatically after each run

After running this cell it will tell you whether to run Cell 2 or skip it.

In [ ]:
import os

# ── Mount Google Drive ─────────────────────────────────────────────
# Drive stores your datasets and saves all model checkpoints and results
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ValueError:
    print('Drive already mounted, continuing...')

# Verify Drive mounted and check if datasets are already saved
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted successfully')
    asvspoof_ready = os.path.exists('/content/drive/MyDrive/fsat_replication/asvspoof')
    fakeaudio_ready = os.path.exists('/content/drive/MyDrive/fsat_replication/fakeaudio')
    if asvspoof_ready and fakeaudio_ready:
        print('✅ Datasets found on Drive → skip Cell 2, go straight to Cell 3')
    else:
        print('⚠️  Datasets not on Drive yet → run Cell 2 to download (~55 min)')
else:
    print('❌ Drive mount failed — try Runtime → Disconnect and reconnect')

# ── Install packages ───────────────────────────────────────────────
!pip install audiomentations soundfile --quiet

import sys, math, random, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
from audiomentations import (
    AddGaussianSNR, AddColorNoise,
    HighPassFilter, LowPassFilter, Gain,
    TanhDistortion, TimeMask, PolarityInversion, Clip
)

warnings.filterwarnings('ignore')
torch.manual_seed(21)
torch.cuda.manual_seed(21)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Running on: {DEVICE}')
if DEVICE == 'cpu':
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → A100')

## Cell 2 — Download Datasets (First Time Only)



Downloads both datasets from Kaggle and saves them to your Google Drive.




*   **First time:** Fill in your Kaggle credentials and run (~55 min)  
*   **Returning user:** Skip this cell entirely and go straight to Cell 3 !!!!

Datasets are read directly from Drive — no copying needed each session

To get your Kaggle credentials:
1. Go to kaggle.com → Settings → Create New API Token
2. Copy your username and token into the fields below

In [ ]:
import os, shutil, json
from tqdm import tqdm

# ── Fill in your Kaggle credentials ───────────────────────────────
KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME_HERE"  # ← replace here
KAGGLE_TOKEN    = "YOUR_KAGGLE_TOKEN_HERE"      # ← replace here

# ── Download from Kaggle ───────────────────────────────────────────
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!pip install kaggle --quiet
print("Downloading ASVspoof 2019 (~23 GB, ~25 min)...")
!kaggle datasets download -d awsaf49/asvpoof-2019-dataset -p /content/data/asvspoof --unzip
print("Downloading FakeAudio (~27 GB, ~25 min)...")
!kaggle datasets download -d walimuhammadahmad/fakeaudio -p /content/data/fakeaudio --unzip

# ── Save to Drive permanently ──────────────────────────────────────
def copy_with_progress(src, dst, label):
    all_files = []
    for dirpath, _, filenames in os.walk(src):
        for f in filenames:
            all_files.append(os.path.join(dirpath, f))
    print(f"\n {label}: {len(all_files)} files")
    os.makedirs(dst, exist_ok=True)
    with tqdm(total=len(all_files), unit='file', desc=label) as pbar:
        for src_file in all_files:
            rel = os.path.relpath(src_file, src)
            dst_file = os.path.join(dst, rel)
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)
            shutil.copy2(src_file, dst_file)
            pbar.update(1)
    print(f"✅ {label} saved to Drive!")

os.makedirs('/content/drive/MyDrive/fsat_replication', exist_ok=True)
copy_with_progress('/content/data/asvspoof',
                   '/content/drive/MyDrive/fsat_replication/asvspoof',
                   'ASVspoof → Drive')
copy_with_progress('/content/data/fakeaudio',
                   '/content/drive/MyDrive/fsat_replication/fakeaudio',
                   'FakeAudio → Drive')
print("\n Done! Skip this cell next time, datasets read directly from Drive.")

## Cell 3 — Paths

Points all dataset paths directly at Google Drive.
No local copying needed, audio is read straight from Drive.
All outputs (models, plots, results) also save to Drive automatically.

> **Note:** `MODEL_PT_PATH = None` because the pretrained weights (model.pt)
> are not publicly available. The model trains from scratch instead.

In [ ]:
import os

# ── Point directly at Drive — no copying needed ───────────────────
ASVSPOOF_ROOT  = '/content/drive/MyDrive/fsat_replication/asvspoof'
FAKEAUDIO_ROOT = '/content/drive/MyDrive/fsat_replication/fakeaudio'
MODEL_PT_PATH  = None
OUTPUT_DIR     = '/content/drive/MyDrive/fsat_replication'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Audio constants
SAMPLE_RATE  = 16000
MAX_FRAMES   = 600
MAX_SAMPLES  = MAX_FRAMES * 160 + 240

# ASVspoof paths
ASV_TRAIN_AUDIO = f'{ASVSPOOF_ROOT}/LA/LA/ASVspoof2019_LA_train/flac'
ASV_TRAIN_PROTO = f'{ASVSPOOF_ROOT}/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_EVAL_AUDIO  = f'{ASVSPOOF_ROOT}/LA/LA/ASVspoof2019_LA_eval/flac'
ASV_EVAL_PROTO  = f'{ASVSPOOF_ROOT}/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'

# Verify paths
for name, path in [('Train audio', ASV_TRAIN_AUDIO),
                   ('Train proto', ASV_TRAIN_PROTO),
                   ('Eval audio',  ASV_EVAL_AUDIO),
                   ('Eval proto',  ASV_EVAL_PROTO)]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'{status} {name}: {path}')

## Cell 4 — RawNet3 Model

This is an implementation of RawNet3 for deepfake detection based on the authors'
supplementary code (train.py).

**Architecture:**
- Sinc filterbank front-end (128 learnable band-pass filters)
- 3 × Bottle2neck residual blocks with Squeeze-and-Excitation attention
- Global statistics pooling (mean + std → 256-d embedding)
- Binary classifier head (Real / Fake)

**Note:** Paper initializes from pretrained weights (model.pt) which are not publicly available. This replication trains from scratch.

In [ ]:
class SEModule(nn.Module):
    def __init__(self, channels, bottleneck=128):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, bottleneck),
            nn.ReLU(),
            nn.Linear(bottleneck, channels),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x).unsqueeze(-1)


class Bottle2neck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None,
                 baseWidth=26, scale=4, stype='normal', se=True):
        super().__init__()
        width = int(math.floor(planes * (baseWidth / 64.0)))
        self.conv1 = nn.Conv1d(inplanes, width * scale, 1, bias=False)
        self.bn1   = nn.BatchNorm1d(width * scale)
        self.nums  = scale - 1
        self.width = width
        self.scale = scale
        self.stype = stype

        convs, bns = [], []
        for _ in range(self.nums):
            convs.append(nn.Conv1d(width, width, 3,
                                   stride=stride, padding=1, bias=False))
            bns.append(nn.BatchNorm1d(width))
        self.convs = nn.ModuleList(convs)
        self.bns   = nn.ModuleList(bns)

        self.conv3 = nn.Conv1d(width * scale, planes * self.expansion, 1, bias=False)
        self.bn3   = nn.BatchNorm1d(planes * self.expansion)
        self.relu  = nn.ReLU(inplace=True)
        self.se    = SEModule(planes * self.expansion) if se else nn.Identity()
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))

        # Split into scale branche, each branch is width channels
        spx = torch.split(out, self.width, 1)
        outs = []
        sp = spx[0].contiguous()
        sp = self.relu(self.bns[0](self.convs[0](sp)))
        outs.append(sp)

        for i in range(1, self.nums):
            if self.stype == 'stage':
                sp = spx[i].contiguous()
            else:
                sp = sp + spx[i].contiguous()
            sp = self.relu(self.bns[i](self.convs[i](sp)))
            outs.append(sp)

        # Last split goes through unchanged, must match spatial dim of others
        # Use adaptive pool to match if stride caused size mismatch
        last = spx[self.nums]
        if outs[-1].shape[-1] != last.shape[-1]:
            last = F.adaptive_avg_pool1d(last, outs[-1].shape[-1])
        outs.append(last)

        out = torch.cat(outs, dim=1)
        out = self.se(self.bn3(self.conv3(out)))
        if self.downsample is not None:
            residual = self.downsample(x)
        return self.relu(out + residual)


class RawNet3_detect(nn.Module):
    def __init__(self, nOut=256, n_cls=2, sinc_stride=10,
                 log_sinc=True, norm_sinc=True, model_scale=8, **kwargs):
        super().__init__()
        self.log_sinc  = log_sinc
        self.norm_sinc = norm_sinc

        self.conv1 = nn.Conv1d(1, 128, 251, stride=sinc_stride, padding=125, bias=False)
        self.bn1   = nn.BatchNorm1d(128)

        self.layer1 = self._make_layer(128, 32, model_scale, stride=3)
        self.layer2 = self._make_layer(128, 32, model_scale, stride=3)
        self.layer3 = self._make_layer(128, 32, model_scale, stride=3)

        self.fc      = nn.Linear(256, nOut)
        self.bn_out  = nn.BatchNorm1d(nOut)
        self.classifier = nn.Linear(nOut, n_cls)

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, inplanes, planes, scale, stride):
        downsample = nn.Sequential(
            nn.Conv1d(inplanes, planes * Bottle2neck.expansion,
                      1, stride=stride, bias=False),
            nn.BatchNorm1d(planes * Bottle2neck.expansion),
        )
        return nn.Sequential(
            Bottle2neck(inplanes, planes, stride=stride,
                        downsample=downsample, scale=scale, stype='stage'),
            Bottle2neck(planes * Bottle2neck.expansion, planes, scale=scale),
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = self.conv1(x)
        if self.log_sinc:
            x = torch.log(x.abs() + 1e-6)
        if self.norm_sinc:
            mean = x.mean(dim=-1, keepdim=True)
            std  = x.std(dim=-1, keepdim=True)
            x    = (x - mean) / (std + 1e-8)
        x = F.relu(self.bn1(x))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        mean = x.mean(dim=-1)
        std  = x.std(dim=-1)
        x    = torch.cat([mean, std], dim=1)
        x = F.relu(self.bn_out(self.fc(x)))
        return self.classifier(x)


def load_model(model_pt_path=None, n_cls=2, device=DEVICE):
    model = RawNet3_detect(nOut=256, n_cls=n_cls, sinc_stride=10,
                           log_sinc=True, norm_sinc=True, model_scale=8)
    if model_pt_path and os.path.exists(model_pt_path):
        ckpt  = torch.load(model_pt_path, map_location='cpu')
        state = ckpt.get('model', ckpt)
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f'Loaded pretrained weights — missing: {len(missing)}, unexpected: {len(unexpected)}')
    else:
        print('Training from scratch (no pretrained weights)')
    return model.to(device)


# Smoke test
with torch.no_grad():
    dummy = torch.randn(2, MAX_SAMPLES).to(DEVICE)
    m     = RawNet3_detect().to(DEVICE)
    out   = m(dummy)
    assert out.shape == (2, 2), f"Expected (2,2) got {out.shape}"
print(f'RawNet3_detect output shape: {out.shape}  (batch=2, classes=2)')

## Cell 5 — Datasets

Builds three balanced datasets with zero data leakage:
- **Training set** : 5,000 real + 5,000 fake (balanced to fix class imbalance)
- **Clean eval set** : 4,935 real + 4,935 fake (ASVspoof eval, no overlap with training)
- **OOD test set** : 2,000 real (ASVspoof eval) + 2,000 fake (FakeAudio, unseen synthesis)

The 2,420 eval files borrowed for training are tracked and excluded from all test sets.

In [ ]:
def load_and_pad(path, max_samples=MAX_SAMPLES, sr=SAMPLE_RATE):
    try:
        audio, orig_sr = librosa.load(path, sr=sr, mono=True)
    except Exception:
        return np.zeros(max_samples, dtype=np.float32)
    if len(audio) < max_samples:
        audio = np.pad(audio, (0, max_samples - len(audio)))
    else:
        audio = audio[:max_samples]
    return audio.astype(np.float32)


class BalancedASVspoofDataset(Dataset):
    """
    Balanced training dataset: 5000 real + 5000 fake.

    Real samples:
      - All 2,580 from train set
      - 2,420 borrowed from eval set (tracked to avoid test overlap)
    Fake samples:
      - 5,000 randomly selected from train set's 22,800

    Returns borrowed_eval_ids so the eval loader can exclude them.
    """
    def __init__(self, train_audio, train_proto,
                       eval_audio,  eval_proto,
                       n_per_class=5000, seed=42):
        random.seed(seed)
        self.samples = []
        self.borrowed_eval_ids = set()  # track which eval files we used

        # ── Collect all real + fake from train ──
        train_real, train_fake = [], []
        with open(train_proto) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                fpath = os.path.join(train_audio, parts[1] + '.flac')
                if not os.path.isfile(fpath): continue
                if parts[4] == 'bonafide':
                    train_real.append((fpath, 0))
                else:
                    train_fake.append((fpath, 1))

        # ── Collect real from eval to supplement ──
        eval_real = []
        with open(eval_proto) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                if parts[4] != 'bonafide': continue
                fpath = os.path.join(eval_audio, parts[1] + '.flac')
                if not os.path.isfile(fpath): continue
                eval_real.append((fpath, 0, parts[1]))  # keep filename for tracking

        # ── Build balanced set ──
        random.shuffle(train_real)
        random.shuffle(train_fake)
        random.shuffle(eval_real)

        # Real: all train real + borrow from eval to reach n_per_class
        real_samples = train_real[:n_per_class]
        n_needed = n_per_class - len(real_samples)
        if n_needed > 0:
            borrowed = eval_real[:n_needed]
            for fpath, label, fname in borrowed:
                real_samples.append((fpath, label))
                self.borrowed_eval_ids.add(fname)  # mark as used

        # Fake: n_per_class from train
        fake_samples = train_fake[:n_per_class]

        self.samples = real_samples + fake_samples
        random.shuffle(self.samples)

        print(f'Balanced training set:')
        print(f'  Real : {len(real_samples)} '
              f'({len(train_real[:n_per_class])} from train, '
              f'{len(self.borrowed_eval_ids)} borrowed from eval)')
        print(f'  Fake : {len(fake_samples)}')
        print(f'  Total: {len(self.samples)}')
        print(f'  Borrowed eval IDs: {len(self.borrowed_eval_ids)} '
              f'(excluded from eval set)')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return torch.tensor(load_and_pad(path)), torch.tensor(label, dtype=torch.long)


class ASVspoofEvalDataset(Dataset):
    """
    Eval dataset that excludes any files borrowed for training.
    Ensures zero overlap between train and eval.
    """
    def __init__(self, audio_dir, protocol_file,
                 excluded_ids=None, max_per_class=99999):
        self.samples = []
        excluded_ids = excluded_ids or set()
        real, fake = [], []

        with open(protocol_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                fname = parts[1]
                if fname in excluded_ids: continue  # skip borrowed files
                fpath = os.path.join(audio_dir, fname + '.flac')
                if not os.path.isfile(fpath): continue
                if parts[4] == 'bonafide':
                    real.append((fpath, 0))
                else:
                    fake.append((fpath, 1))

        self.samples = real[:max_per_class] + fake[:max_per_class]
        random.shuffle(self.samples)

        print(f'Eval set (no overlap): '
              f'{len(real[:max_per_class])} real, '
              f'{len(fake[:max_per_class])} fake, '
              f'{len(excluded_ids)} excluded')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return torch.tensor(load_and_pad(path)), torch.tensor(label, dtype=torch.long)


class OODTestDataset(Dataset):
    """
    Balanced OOD test set:
    - Real audio from ASVspoof eval (not used in training)
    - Fake audio from FakeAudio (unseen synthesis methods)
    """
    def __init__(self, asv_audio_dir, asv_proto,
                 fakeaudio_root, excluded_ids=None, n_samples=2000):
        self.samples = []
        excluded_ids = excluded_ids or set()

        # Real from ASVspoof eval (excluding borrowed)
        real_files = []
        with open(asv_proto) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5 and parts[4] == 'bonafide':
                    if parts[1] in excluded_ids: continue
                    fpath = os.path.join(asv_audio_dir, parts[1] + '.flac')
                    if os.path.isfile(fpath):
                        real_files.append(fpath)
        random.shuffle(real_files)
        for f in real_files[:n_samples]:
            self.samples.append((f, 0))

        # Fake from FakeAudio
        gen_dir = os.path.join(fakeaudio_root, 'generated_audio')
        fake_files = []
        if os.path.exists(gen_dir):
            for model_folder in os.listdir(gen_dir):
                model_path = os.path.join(gen_dir, model_folder)
                audio_path = os.path.join(model_path, 'generated')
                if not os.path.isdir(audio_path):
                    audio_path = model_path
                if os.path.isdir(audio_path):
                    fake_files += [
                        os.path.join(audio_path, f)
                        for f in os.listdir(audio_path)
                        if f.endswith('.wav')]
        random.shuffle(fake_files)
        for f in fake_files[:n_samples]:
            self.samples.append((f, 1))

        real = sum(1 for _, l in self.samples if l == 0)
        print(f'OOD test set: {len(self.samples)} samples '
              f'({real} real, {len(self.samples)-real} fake)')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return torch.tensor(load_and_pad(path)), torch.tensor(label, dtype=torch.long)


# ── Build all datasets ─────────────────────────────────────────────
print('Loading datasets...\n')

# Balanced training set
train_ds = BalancedASVspoofDataset(
    ASV_TRAIN_AUDIO, ASV_TRAIN_PROTO,
    ASV_EVAL_AUDIO,  ASV_EVAL_PROTO,
    n_per_class=5000)

# Eval set — balanced, excludes borrowed files
asv_eval_ds = ASVspoofEvalDataset(
    ASV_EVAL_AUDIO, ASV_EVAL_PROTO,
    excluded_ids=train_ds.borrowed_eval_ids,
    max_per_class=4935)  # ← capped to match real count

# OOD test set
ood_ds = OODTestDataset(
    ASV_EVAL_AUDIO, ASV_EVAL_PROTO,
    FAKEAUDIO_ROOT,
    excluded_ids=train_ds.borrowed_eval_ids)

BATCH = 16
train_loader = DataLoader(train_ds,    batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True)
asv_loader   = DataLoader(asv_eval_ds, batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)
ood_loader   = DataLoader(ood_ds,      batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'\nTrain batches : {len(train_loader)}')
print(f'ASV eval batch: {len(asv_loader)}')
print(f'OOD batches   : {len(ood_loader)}')

## Cell 6 — Utilities
Learning rate scheduler, mixup regularization, and evaluation metrics.
Ported directly from the authors' lr_schedule.py and regularization.py.

In [ ]:
# ── LR scheduler (authors' lr_schedule.py) ───────────────────────
class WarmUpCosineLR(torch.optim.lr_scheduler._LRScheduler):
    """Linear warmup then cosine annealing — matches paper's schedule exactly."""
    def __init__(self, optimizer, warmup_steps, total_steps,
                 warmup_lr=1e-6, max_lr=1e-5, min_lr=1e-7, last_step=-1):
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.warmup_lr    = warmup_lr
        self.max_lr       = max_lr
        self.min_lr       = min_lr
        super().__init__(optimizer, last_step)

    def get_lr(self):
        if self.last_epoch < self.warmup_steps:
            lr = self.warmup_lr + (self.max_lr - self.warmup_lr) * (
                self.last_epoch / max(1, self.warmup_steps))
        else:
            t = (self.last_epoch - self.warmup_steps) / (
                self.total_steps - self.warmup_steps)
            lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (
                1 + math.cos(math.pi * t))
        return [lr for _ in self.optimizer.param_groups]


# ── Mixup (authors' regularization.py) ───────────────────────────
def mixup_data(x, y, alpha=0.5):
    lam   = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1.0
    idx   = torch.randperm(x.size(0))
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ── Metrics ───────────────────────────────────────────────────────
def compute_metrics(labels, preds):
    labels, preds = np.array(labels), np.array(preds)
    real_mask = labels == 0
    fake_mask = labels == 1
    real_acc = accuracy_score(labels[real_mask], preds[real_mask]) if real_mask.any() else 0.
    fake_acc = accuracy_score(labels[fake_mask], preds[fake_mask]) if fake_mask.any() else 0.
    f1       = f1_score(labels, preds, average='binary', pos_label=1, zero_division=0)
    return real_acc, fake_acc, f1

def evaluate(model, loader, desc='Eval'):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc, leave=False):
            x = x.to(DEVICE)
            preds = model(x).argmax(1)
            all_labels.extend(y.tolist())
            all_preds.extend(preds.cpu().tolist())
    return compute_metrics(all_labels, all_preds)

print('Utilities ready')

## Cell 7 — RandAugment for audio
Port of the authors' Figure 10 / `dataset_aug_all.py`. Selects N random transforms with probability p.

In [ ]:
class AudioRandAugment:
    """
    RandAugment for audio — replicates Figure 10 from the paper.

    At each training step, randomly picks N transforms from the
    taxonomy and applies each with probability p.
    Paper values: N=1, p=0.9 (from readme.txt and hyperparameters.py)
    """
    # Full transform list from the paper's taxonomy (Figure 9)
    TRANSFORMS = [
    AddGaussianSNR(min_snr_db=5, max_snr_db=40, p=1.0),
    AddColorNoise(min_snr_db=5, max_snr_db=40, p=1.0),
    HighPassFilter(min_cutoff_freq=200, max_cutoff_freq=2400, p=1.0),
    LowPassFilter(min_cutoff_freq=150, max_cutoff_freq=4000, p=1.0),
    Gain(min_gain_db=-12, max_gain_db=12, p=1.0),
    Clip(a_min=-0.9, a_max=0.9, p=1.0),
    TanhDistortion(min_distortion=0.01, max_distortion=0.7, p=1.0),
    TimeMask(min_band_part=0.0, max_band_part=0.5, p=1.0),
    PolarityInversion(p=1.0),
    ]

    def __init__(self, N=1, p=0.9, sr=SAMPLE_RATE):
        self.N  = N
        self.p  = p
        self.sr = sr

    def __call__(self, audio_tensor):
        """audio_tensor: torch.Tensor shape (T,) — returns augmented (T,)"""
        x = audio_tensor.cpu().numpy()  # audiomentations works on numpy
        chosen = random.choices(self.TRANSFORMS, k=self.N)
        for t in chosen:
            if random.random() < self.p:
                try:
                    x = t(samples=x, sample_rate=self.sr)
                except Exception:
                    pass  # skip transforms that fail on edge cases
        return torch.tensor(x, dtype=torch.float32)


def apply_randaug_batch(x_batch, augmenter):
    """Apply RandAugment to each sample, ensuring consistent output length."""
    results = []
    for i in range(x_batch.size(0)):
        aug = augmenter(x_batch[i])
        # Pad or truncate to match MAX_SAMPLES exactly
        if aug.shape[-1] < MAX_SAMPLES:
            aug = F.pad(aug, (0, MAX_SAMPLES - aug.shape[-1]))
        else:
            aug = aug[..., :MAX_SAMPLES]
        results.append(aug)
    return torch.stack(results).to(DEVICE)


print('AudioRandAugment ready')

## Cell 8 — F-SAT: Frequency-Selective Adversarial Training

Direct port of the authors' `adversarial/frequency_attack.py` (`attack_spec` function)  
and the training loop from `train_attack_frequency.py`.

Key equations from paper Section 4.1:
- Perturb only the **magnitude** of STFT bins above `target_freq` (default 4000 Hz)
- Phase is preserved exactly
- Total loss = `L_clean + γ * L_robust` with `γ=0.1`

In [ ]:
def spectrum_attack(model, audio, label,
                    epsilon=0.005, alpha=0.002,
                    attack_iters=2, restarts=1,
                    target_freq=4000,
                    n_fft=1024, hop_length=512, win_length=1024,
                    sr=SAMPLE_RATE):
    """
    Frequency-Selective adversarial attack.
    Direct port of attack_spec() from the authors' eval_frequency_attack.py.

    Perturbs STFT magnitude bins from target_freq to Nyquist (4k–8k Hz)
    using PGD. Phase is preserved. Converts back to waveform via iSTFT.

    Args:
        model:       the RawNet3_detect model
        audio:       (B, T) waveform tensor on DEVICE
        label:       (B,) ground-truth labels on DEVICE
        target_freq: lower bound of attacked frequency range (Hz)
    Returns:
        perturbed audio (B, T) on DEVICE
    """
    model.eval()
    window = torch.hann_window(win_length).to(DEVICE)

    # STFT → magnitude + phase
    spec = torch.stft(audio, n_fft=n_fft, hop_length=hop_length,
                      win_length=win_length, window=window,
                      center=True, normalized=False,
                      onesided=True, return_complex=True)  # (B, F, frames)
    magnitude = spec.abs()   # (B, F, frames)
    phase     = spec.angle() # (B, F, frames)

    B, num_freq, num_frames = magnitude.shape
    # Frequency bin index corresponding to target_freq
    # (matches authors: index = ceil(target_freq * n_fft / sr))
    freq_idx = int(np.ceil(target_freq * n_fft / sr))
    n_attack = num_freq - freq_idx  # number of bins to perturb

    max_loss  = torch.zeros(B, device=DEVICE)
    max_delta = torch.zeros(B, n_attack, num_frames, device=DEVICE)

    for _ in range(restarts):
        delta = torch.zeros(B, n_attack, num_frames, device=DEVICE)
        delta = delta.uniform_(-epsilon, epsilon).requires_grad_(True)

        for _ in range(attack_iters):
            # Apply perturbation to high-freq bins only
            perturbed_mag = magnitude.clone()
            perturbed_mag[:, freq_idx:, :] = perturbed_mag[:, freq_idx:, :] + delta

            # Reconstruct complex STFT and invert
            perturbed_spec = perturbed_mag * torch.exp(1j * phase)
            new_audio = torch.istft(perturbed_spec, n_fft=n_fft,
                                    hop_length=hop_length, win_length=win_length,
                                    window=window, center=True,
                                    normalized=False, onesided=True)
            # Forward + loss
            loss = F.cross_entropy(model(new_audio), label)
            loss.backward()

            # PGD step
            with torch.no_grad():
                delta_new = delta + alpha * torch.sign(delta.grad)
                delta_new = torch.clamp(delta_new, -epsilon, epsilon)
            delta = delta_new.detach().requires_grad_(True)

        # Keep the worst-case delta across restarts
        with torch.no_grad():
            all_loss = F.cross_entropy(
                model(torch.istft(
                    (magnitude.clone().index_add_(
                        1,
                        torch.arange(freq_idx, num_freq, device=DEVICE),
                        delta.detach()
                    )) * torch.exp(1j * phase),
                    n_fft=n_fft, hop_length=hop_length, win_length=win_length,
                    window=window, center=True, normalized=False, onesided=True)),
                label, reduction='none')
        update = all_loss >= max_loss
        max_delta[update] = delta.detach()[update]
        max_loss = torch.max(max_loss, all_loss)

    # Build final perturbed audio
    with torch.no_grad():
        final_mag = magnitude.clone()
        final_mag[:, freq_idx:, :] += max_delta
        final_audio = torch.istft(
            final_mag * torch.exp(1j * phase),
            n_fft=n_fft, hop_length=hop_length, win_length=win_length,
            window=window, center=True, normalized=False, onesided=True)

    model.train()
    return final_audio.detach()


print('spectrum_attack ready')

## Cell 9 — Training loops

Three training functions matching the paper's three conditions in Table 3.  
All use the **exact hyperparameters from `readme.txt`**:
```
lr=1e-5, epochs=15, mixup, mixup_alpha=0.5, max_frames=600,
bs=16, aug_num=1, aug_prob=0.9, attack=spectrum,
attack_iters=2, spectrum_target_freq=4000,
spectrum_epsilon=0.01, spectrum_alpha=0.005, gamma=0.1
```

In [ ]:
# ── Shared hyperparameters ─────────────────────────────────────────
HP = dict(
    epochs        = 15,    # 1. increased from 5 → 15
    lr            = 1e-5,
    warmup_lr     = 1e-6,
    min_lr        = 1e-7,
    warmup_epochs = 1,
    mixup_alpha   = 0.5,
    aug_N         = 1,
    aug_p         = 0.9,
    epsilon       = 0.02,  # 2. increased from 0.01 → 0.02
    alpha         = 0.008, # 2. scaled with epsilon
    attack_iters  = 4,     # 2. increased from 2 → 4
    restarts      = 2,     # 2. added restarts
    target_freq   = 4000,  # will extend to 2kHz in F-SAT (improvement 4)
    gamma         = 0.3,   # 2. increased from 0.1 → 0.3
)


def make_scheduler(optimizer, train_loader, epochs):
    total_steps  = len(train_loader) * epochs
    warmup_steps = len(train_loader) * HP['warmup_epochs']
    return WarmUpCosineLR(optimizer, warmup_steps, total_steps,
                          HP['warmup_lr'], HP['lr'], HP['min_lr'])


def log(history, epoch, loss, real_acc, fake_acc, f1):
    history.append(dict(epoch=epoch, loss=loss,
                        real_acc=real_acc, fake_acc=fake_acc, f1=f1))
    print(f'  Ep {epoch:02d} | loss={loss:.4f} | '
          f'real={real_acc:.3f} fake={fake_acc:.3f} F1={f1:.3f}')


def cross_class_mixup(x, y, alpha=0.5):
    """
    Improvement 6: Cross-class mixup.
    Blends samples across class boundaries to smooth the decision boundary.
    Forces model to learn gradual real→fake spectrum rather than memorizing extremes.
    """
    lam = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1.0
    # Shuffle so real mixes with fake and vice versa
    idx   = torch.randperm(x.size(0))
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam


# ── Variant 1: Baseline ───────────────────────────────────────────
def train_baseline(model, train_loader, val_loader, save_path):
    """
    RawNet3 baseline with improvements:
    - Class weighted loss (improvement 3)
    - Cross-class mixup (improvement 6)
    - 15 epochs (improvement 1)
    """
    # 3. Class weighted loss — extra protection even with balanced data
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([1.5, 1.0]).to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'])
    scheduler = make_scheduler(optimizer, train_loader, HP['epochs'])
    history   = []

    for epoch in range(1, HP['epochs'] + 1):
        model.train()
        total_loss = 0.
        for x, y in tqdm(train_loader, desc=f'[Baseline] Ep {epoch}', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            # 6. Cross-class mixup
            x, ya, yb, lam = cross_class_mixup(x, y, HP['mixup_alpha'])
            optimizer.zero_grad()
            loss = lam * criterion(model(x), ya) + (1 - lam) * criterion(model(x), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
        r, f, f1 = evaluate(model, val_loader, 'Val')
        log(history, epoch, total_loss / len(train_loader), r, f, f1)

    torch.save({'model_state_dict': model.state_dict()}, save_path)
    print(f'  Saved → {save_path}')
    return history


# ── Variant 2: + RandAugment ──────────────────────────────────────
def train_randaug(model, train_loader, val_loader, save_path):
    """
    RawNet3 + RandAug with improvements:
    - Class weighted loss
    - Cross-class mixup
    - 15 epochs
    """
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([1.5, 1.0]).to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'])
    scheduler = make_scheduler(optimizer, train_loader, HP['epochs'])
    augmenter = AudioRandAugment(N=HP['aug_N'], p=HP['aug_p'])
    history   = []

    for epoch in range(1, HP['epochs'] + 1):
        model.train()
        total_loss = 0.
        for x, y in tqdm(train_loader, desc=f'[+RandAug] Ep {epoch}', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            x    = apply_randaug_batch(x, augmenter)
            # 6. Cross-class mixup (didnt end up helping)
            x, ya, yb, lam = cross_class_mixup(x, y, HP['mixup_alpha'])
            optimizer.zero_grad()
            loss = lam * criterion(model(x), ya) + (1 - lam) * criterion(model(x), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
        r, f, f1 = evaluate(model, val_loader, 'Val')
        log(history, epoch, total_loss / len(train_loader), r, f, f1)

    torch.save({'model_state_dict': model.state_dict()}, save_path)
    print(f'  Saved → {save_path}')
    return history


# ── Variant 3: + RandAug + F-SAT ─────────────────────────────────
def train_fsat(model, train_loader, val_loader, save_path):
    """
    RawNet3 + RandAug + F-SAT with ALL improvements:
    - 15 epochs (this helped)
    - Stronger PGD: epsilon=0.02, alpha=0.008, iters=4, restarts=2 (improvement 2)
    - Class weighted loss (improvement 3)
    - Extended frequency range to 2kHz (didnt help, woulbe been better 4-8kHz)
    - Cross-class mixup (this didnt end up helping)
    """
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([1.5, 1.0]).to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=HP['lr'])
    scheduler = make_scheduler(optimizer, train_loader, HP['epochs'])
    augmenter = AudioRandAugment(N=HP['aug_N'], p=HP['aug_p'])
    history   = []

    for epoch in range(1, HP['epochs'] + 1):
        model.train()
        total_loss = 0.
        for x, y in tqdm(train_loader, desc=f'[+F-SAT]   Ep {epoch}', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)

            # Step 1 — augment
            x_aug = apply_randaug_batch(x, augmenter)

            # Step 2 — F-SAT with stronger attack (improvements 2 + 4)
            # 4. Extended to 2kHz instead of 4kHz for wider frequency coverage
            x_adv = spectrum_attack(
                model, x_aug, y,
                epsilon=HP['epsilon'],
                alpha=HP['alpha'],
                attack_iters=HP['attack_iters'],
                restarts=HP['restarts'],
                target_freq=2000)  # 4. extended from 4000 → 2000 Hz

            # Step 3 — cross-class mixup on clean and attacked separately
            x_clean, ya_c, yb_c, lam_c = cross_class_mixup(x_aug, y, HP['mixup_alpha'])
            x_adv,   ya_a, yb_a, lam_a = cross_class_mixup(x_adv, y, HP['mixup_alpha'])

            # Step 4 — combined loss (improvement 3: weighted criterion)
            model.train()
            optimizer.zero_grad()
            loss_clean  = lam_c * criterion(model(x_clean), ya_c) + \
                         (1 - lam_c) * criterion(model(x_clean), yb_c)
            loss_robust = lam_a * criterion(model(x_adv), ya_a) + \
                         (1 - lam_a) * criterion(model(x_adv), yb_a)
            loss = loss_clean + HP['gamma'] * loss_robust
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item() / (1 + HP['gamma'])

        r, f, f1 = evaluate(model, val_loader, 'Val')
        log(history, epoch, total_loss / len(train_loader), r, f, f1)

    torch.save({'model_state_dict': model.state_dict()}, save_path)
    print(f'  Saved → {save_path}')
    return history


print('Training functions ready (with improvements 1, 2, 3, 4, 6)')

## Cell 10 — Train all three models
At 15 epochs each:
- Baseline
- +RandAug
- +F-SAT

Checkpoints save to Google Drive after each variant.

### Run to prevent colab from disconnecting

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

### Training

In [ ]:
histories = {}

print('='*60)
print('[1/3] BASELINE — RawNet3 + improvements')
print('='*60)
model_baseline = load_model(MODEL_PT_PATH)
histories['baseline'] = train_baseline(
    model_baseline, train_loader, asv_loader,
    save_path=os.path.join(OUTPUT_DIR, 'model_baseline_v1.pth'))

print('\n' + '='*60)
print('[2/3] RANDAUG — RawNet3 + RandAug + improvements')
print('='*60)
model_randaug = load_model(MODEL_PT_PATH)
histories['randaug'] = train_randaug(
    model_randaug, train_loader, asv_loader,
    save_path=os.path.join(OUTPUT_DIR, 'model_randaug_v1.pth'))

print('\n' + '='*60)
print('[3/3] F-SAT — RawNet3 + RandAug + F-SAT + improvements')
print('='*60)
model_fsat = load_model(MODEL_PT_PATH)
histories['fsat'] = train_fsat(
    model_fsat, train_loader, asv_loader,
    save_path=os.path.join(OUTPUT_DIR, 'model_fsat_v1.pth'))

with open(os.path.join(OUTPUT_DIR, 'histories_v1.json'), 'w') as f:
    json.dump(histories, f, indent=2)
print('\n✅ All three models trained and saved!')

This verifies if the models were saved:



In [ ]:
import os
for f in ['model_baseline_v1.pth', 'model_randaug_v1.pth', 'model_fsat_v1.pth', 'histories_v1.json']:
    path = f'/content/drive/MyDrive/fsat_replication/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f'FOUND IN DRIVE!! {f}: {size:.1f} MB')
    else:
        print(f'❌ {f}: NOT FOUND')

## Cell 11 — Evaluation (Table 3 replication)

Evaluates each model on four conditions matching the paper's Table 3:

| Condition | What it tests |
|---|---|
| **Original** | Clean ASVspoof2019 eval set |
| **OOD** | FakeAudio Kaggle dataset (newer, unseen synthesis methods) |
| **Attack (Time)** | PGD on raw waveform |
| **Attack (Freq)** | F-SAT attack on STFT magnitude, 4–8kHz |

In [ ]:
def eval_under_pgd_time(model, loader, epsilon=1e-4, alpha=4e-5, iters=2):
    """PGD attack on raw waveform — matches eval_pgd.py."""
    model.eval()
    all_labels, all_preds = [], []
    for x, y in tqdm(loader, desc='PGD-Time', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        delta = torch.zeros_like(x).uniform_(-epsilon, epsilon).requires_grad_(True)
        for _ in range(iters):
            loss = F.cross_entropy(model(x + delta), y)
            loss.backward()
            with torch.no_grad():
                delta = torch.clamp(delta + alpha * delta.grad.sign(),
                                    -epsilon, epsilon)
            delta = delta.detach().requires_grad_(True)
        with torch.no_grad():
            preds = model(torch.clamp(x + delta, -1, 1)).argmax(1)
        all_labels.extend(y.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
    return compute_metrics(all_labels, all_preds)


def eval_under_freq_attack(model, loader):
    """Frequency-domain attack — uses the same spectrum_attack as training."""
    model.eval()
    all_labels, all_preds = [], []
    for x, y in tqdm(loader, desc='Freq-Attack', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        x_adv = spectrum_attack(model, x, y,
                                epsilon=HP['epsilon'], alpha=HP['alpha'],
                                attack_iters=HP['attack_iters'],
                                target_freq=HP['target_freq'])
        with torch.no_grad():
            preds = model(x_adv).argmax(1)
        all_labels.extend(y.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
    return compute_metrics(all_labels, all_preds)


# Run full evaluation
models = {
    'RawNet3 (Baseline)':        model_baseline,
    'RawNet3 + RandAug':         model_randaug,
    'RawNet3 + RandAug + F-SAT': model_fsat,
}
results = {}

for name, model in models.items():
    print(f'\n{"-"*55}\n{name}\n{"-"*55}')
    results[name] = {}

    r, f, f1 = evaluate(model, asv_loader, 'Clean ASV')
    results[name]['origin'] = {'real': r, 'fake': f, 'f1': f1}
    print(f'  Clean (ASV)  | real={r:.3f} fake={f:.3f} F1={f1:.3f}')

    r, f, f1 = evaluate(model, ood_loader, 'OOD')  # fixed: ood_loader
    results[name]['ood'] = {'real': r, 'fake': f, 'f1': f1}
    print(f'  OOD          | real={r:.3f} fake={f:.3f} F1={f1:.3f}')

    r, f, f1 = eval_under_pgd_time(model, asv_loader)
    results[name]['attack_time'] = {'real': r, 'fake': f, 'f1': f1}
    print(f'  Attack(Time) | real={r:.3f} fake={f:.3f} F1={f1:.3f}')

    r, f, f1 = eval_under_freq_attack(model, asv_loader)
    results[name]['attack_freq'] = {'real': r, 'fake': f, 'f1': f1}
    print(f'  Attack(Freq) | real={r:.3f} fake={f:.3f} F1={f1:.3f}')

with open(os.path.join(OUTPUT_DIR, 'results_v1.json'), 'w') as f:
    json.dump(results, f, indent=2)
print('\n Evaluation complete. Results saved.')

###This is to verify results are saved to the drive:

In [ ]:
import os
path = '/content/drive/MyDrive/fsat_replication/results_v1.json'
if os.path.exists(path):
    size = os.path.getsize(path) / 1e3
    print(f'✅ results.json: {size:.1f} KB — saved to Drive!')
    # Show contents
    import json
    with open(path) as f:
        data = json.load(f)
    for model, conditions in data.items():
        print(f'\n{model}:')
        for cond, metrics in conditions.items():
            print(f'  {cond}: real={metrics["real"]:.3f} fake={metrics["fake"]:.3f} f1={metrics["f1"]:.3f}')
else:
    print('❌ results.json NOT FOUND on Drive')

## Cell 12 — Plots
To see the results represented visually.

In [ ]:
# ── Load training histories from Drive ──────────────────────────────
import json
with open(os.path.join(OUTPUT_DIR, 'histories_v1.json')) as f:
    histories = json.load(f)
print('Loaded histories_v1.json')
print('Keys:', list(histories.keys()))

# ── Plot 1: Training F1 curves  ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['#2196F3', '#FF9800', '#4CAF50']
labels = ['RawNet3 (Baseline)', 'RawNet3 + RandAug', 'RawNet3 + RandAug + F-SAT']
keys   = ['baseline', 'randaug', 'fsat']

for c, lbl, key in zip(colors, labels, keys):
    h = histories[key]
    axes[0].plot([e['epoch'] for e in h], [e['f1']   for e in h], color=c, label=lbl, lw=2)
    axes[1].plot([e['epoch'] for e in h], [e['loss'] for e in h], color=c, label=lbl, lw=2)

for ax, title in zip(axes, ['Validation F1 (clean ASV)', 'Training Loss']):
    ax.set(title=title, xlabel='Epoch')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves_v1.png'), dpi=150)
plt.show()
print('Saved training_curves_v1.png')

# ── Plot 2: Table 3 heatmap ──────────────────────────────
conditions  = ['origin', 'ood', 'attack_time', 'attack_freq']
cond_labels = ['Clean\n(ASV)', 'OOD\n(FakeAudio)', 'Attack\n(Time)', 'Attack\n(Freq)']
model_names = list(results.keys())
matrix      = np.array([[results[m][c]['f1'] for c in conditions] for m in model_names])

fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(matrix, vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(4)); ax.set_xticklabels(cond_labels, fontsize=11)
ax.set_yticks(range(3)); ax.set_yticklabels(model_names, fontsize=10)
ax.set_title('F1 Score by Model × Condition', pad=12)
plt.colorbar(im, ax=ax, label='F1')
for i in range(3):
    for j in range(4):
        ax.text(j, i, f'{matrix[i,j]:.3f}', ha='center', va='center',
                fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'results_table3_v1.png'), dpi=150)
plt.show()
print('Saved results_table3_v1.png')

# ── Plot 3: Figure 2 replication — high-frequency dependence ─────
cutoffs   = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000]
real_acc_list, fake_acc_list = [], []
model_baseline.eval()

for cutoff in cutoffs:
    all_labels, all_preds = [], []
    for x, y in tqdm(asv_loader, desc=f'HiPass {cutoff}Hz', leave=False):
        x = x.to(DEVICE)
        if cutoff > 0:
            win    = max(1, SAMPLE_RATE // cutoff)
            kernel = torch.ones(1, 1, win, device=DEVICE) / win
            lp     = F.conv1d(x.unsqueeze(1), kernel,
                               padding=win//2).squeeze(1)[..., :x.shape[-1]]
            x = x - lp
        with torch.no_grad():
            preds = model_baseline(x).argmax(1)
        all_labels.extend(y.tolist())
        all_preds.extend(preds.cpu().tolist())
    r, f, _ = compute_metrics(all_labels, all_preds)
    real_acc_list.append(r)
    fake_acc_list.append(f)

plt.figure(figsize=(8, 4))
plt.plot(cutoffs, real_acc_list, 'g-o', label='Real accuracy', lw=2)
plt.plot(cutoffs, fake_acc_list, 'r--o', label='Fake accuracy', lw=2)
plt.axvline(4000, color='blue',   ls=':', alpha=0.7, label='4kHz')
plt.axvline(6000, color='orange', ls=':', alpha=0.7, label='6kHz')
plt.xlabel('High-pass cutoff (Hz)')
plt.ylabel('Accuracy')
plt.title('High-frequency dependence — Baseline (Figure 2 replication)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figure2_replication_v1.png'), dpi=150)
plt.show()
print('Saved figure2_replication_v1.png')

print('\n✅ All Run 3 plots saved to Drive!')